# Study Buddy — fine-tuned model API (Colab + FastAPI + ngrok)

Serves the LoRA fine-tuned `Qwen/Qwen2.5-0.5B-Instruct` to the Study Buddy backend (`POST /api/ai-chat` → this notebook's `/generate`).

**Before running, add these in the 🔑 Secrets panel (left sidebar) and switch on "Notebook access" for each:**

| Secret | Required | Value |
|---|---|---|
| `STUDY_BUDDY_API_TOKEN` | yes | the **same** value as `STUDY_BUDDY_API_TOKEN` in `Backend/.env` |
| `NGROK_AUTHTOKEN` | yes | your token from dashboard.ngrok.com → Your Authtoken |
| `NGROK_DOMAIN` | optional | your free static domain (dashboard.ngrok.com → Domains), e.g. `name.ngrok-free.app` — keeps the URL the same every session so `.env` never needs editing |

Then **Runtime → Run all**. GPU (T4) is faster but CPU works for a 0.5B model.

Contract (matches `Backend/routes/aiChat.js`): `POST /generate` with header `Authorization: Bearer <token>` and body `{"message": "..."}` → `{"reply": "..."}`.

In [ ]:
# 1. Dependencies (Colab already ships torch)
import os, sys

# The model is served in fp16/fp32, so bitsandbytes is not needed. Old builds of it crash PEFT
# on Colab's triton 3 ("No module named 'triton.ops'"), so remove it.
!pip uninstall -y -q bitsandbytes
!pip install -q -U transformers peft accelerate fastapi "uvicorn[standard]" pyngrok

# Libraries already imported in this session stay in memory at their old versions (and PEFT
# caches whether bitsandbytes exists). Restart once so the fresh installs are used.
stale = [m for m in ("bitsandbytes", "peft", "transformers", "accelerate") if m in sys.modules]
if stale:
    print(f"Restarting the runtime to unload {stale}.")
    print("When it reconnects, click Runtime → Run all again (no restart will be needed the second time).", flush=True)
    import time; time.sleep(2)  # let the message reach the notebook before the kernel exits
    os.kill(os.getpid(), 9)
print("Dependencies ready")

In [ ]:
# 2. Configuration + secrets (values are never printed)
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_DIR = ""          # leave blank to auto-detect under /content/drive/MyDrive, or set the folder holding adapter_config.json
PORT = 8000
MAX_NEW_TOKENS = 512
MAX_MESSAGE_CHARS = 4000  # same limit as the backend
# None = Qwen's default chat-template system prompt. If you used a custom system prompt
# while fine-tuning, paste that exact text here so inference matches training.
SYSTEM_PROMPT = None

from google.colab import userdata

def read_secret(name, required=True):
    try:
        value = userdata.get(name)
    except Exception as e:
        kind = type(e).__name__
        if "NotebookAccess" in kind:
            raise RuntimeError(f"Secret '{name}' exists but 'Notebook access' is off. Enable it in the 🔑 Secrets panel.") from None
        if not required:
            return ""
        raise RuntimeError(f"Missing Colab secret '{name}'. Add it in the 🔑 Secrets panel and enable Notebook access.") from None
    value = (value or "").strip()
    if required and not value:
        raise RuntimeError(f"Colab secret '{name}' is empty.")
    return value

API_TOKEN = read_secret("STUDY_BUDDY_API_TOKEN")
NGROK_AUTHTOKEN = read_secret("NGROK_AUTHTOKEN")
NGROK_DOMAIN = read_secret("NGROK_DOMAIN", required=False)

# The length lets you compare with Backend/.env (`npm run ai:check` prints the local length) without revealing either value.
print(f"STUDY_BUDDY_API_TOKEN loaded (length {len(API_TOKEN)})")
print("NGROK_AUTHTOKEN loaded")
print("NGROK_DOMAIN:", "set" if NGROK_DOMAIN else "not set (a random URL will be used)")

In [ ]:
# 3. Mount Drive and locate the LoRA adapter
import os, json, glob
from google.colab import drive

drive.mount("/content/drive")

def adapter_files_ok(path):
    has_cfg = os.path.isfile(os.path.join(path, "adapter_config.json"))
    has_weights = any(os.path.isfile(os.path.join(path, f)) for f in ("adapter_model.safetensors", "adapter_model.bin"))
    return has_cfg and has_weights

if not ADAPTER_DIR:
    print("Searching Drive for adapter_config.json (can take a minute)...")
    candidates = sorted(
        {os.path.dirname(p) for p in glob.glob("/content/drive/MyDrive/**/adapter_config.json", recursive=True)},
        key=os.path.getmtime, reverse=True,
    )
    candidates = [c for c in candidates if adapter_files_ok(c)]
    for c in candidates:
        print("  found:", c)
    if not candidates:
        raise RuntimeError("No LoRA adapter (adapter_config.json + adapter_model.*) found in MyDrive. Set ADAPTER_DIR in cell 2.")
    # Prefer a final adapter folder over intermediate trainer checkpoints
    finals = [c for c in candidates if "checkpoint-" not in os.path.basename(c)]
    ADAPTER_DIR = (finals or candidates)[0]
    if len(candidates) > 1:
        print("Multiple adapters found; using the newest final one. Set ADAPTER_DIR in cell 2 to override.")

if not adapter_files_ok(ADAPTER_DIR):
    raise RuntimeError(f"{ADAPTER_DIR} must contain adapter_config.json and adapter_model.safetensors (or .bin).")

with open(os.path.join(ADAPTER_DIR, "adapter_config.json")) as f:
    adapter_cfg = json.load(f)

print("Adapter:", ADAPTER_DIR)
print("  files:", sorted(os.listdir(ADAPTER_DIR)))
for key in ("peft_type", "task_type", "base_model_name_or_path", "r", "lora_alpha", "target_modules", "modules_to_save", "peft_version"):
    print(f"  {key}: {adapter_cfg.get(key)}")
if adapter_cfg.get("peft_type") not in (None, "LORA"):
    raise RuntimeError(f"Expected a LoRA adapter, got {adapter_cfg.get('peft_type')}.")
trained_base = str(adapter_cfg.get("base_model_name_or_path") or "").lower()
if trained_base and "qwen2.5-0.5b" not in trained_base:
    print("⚠️  Adapter base does not look like Qwen2.5-0.5B — weight shapes may not match.")
if "bnb" in trained_base or "4bit" in trained_base:
    print("Adapter was trained on a 4-bit (QLoRA) base; it will be applied to the full-precision base, the standard way to serve it.")

In [ ]:
# 4. Load base model + adapter, with compatibility fixes
import inspect, shutil, sys, threading

# PEFT imports bitsandbytes whenever it is merely installed — even for a non-quantized model —
# and old builds crash on triton 3 ("No module named 'triton.ops'"). Hide it from this session
# and reset the cached availability checks, so loading works even if bitsandbytes is still
# installed or was uninstalled without a restart. Nothing here needs 4/8-bit quantization.
sys.modules["bitsandbytes"] = None
for _mod_name in ("peft.import_utils", "transformers.utils.import_utils", "accelerate.utils.imports"):
    _mod = sys.modules.get(_mod_name)
    for _fn_name in (dir(_mod) if _mod else []):
        if "bnb" in _fn_name or "bitsandbytes" in _fn_name:
            getattr(getattr(_mod, _fn_name), "cache_clear", lambda: None)()

import torch, transformers, peft
from packaging import version
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, LoraConfig

print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
if version.parse(transformers.__version__) < version.parse("4.37"):
    raise RuntimeError("Qwen2.5 needs transformers >= 4.37. Run: !pip install -U transformers, then Runtime → Restart session.")

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print("device:", device)

# Work on a local copy so Drive files are never modified.
LOCAL_ADAPTER = "/content/study_buddy_adapter"
shutil.rmtree(LOCAL_ADAPTER, ignore_errors=True)
shutil.copytree(ADAPTER_DIR, LOCAL_ADAPTER)

# Adapters saved by a newer peft (or Unsloth) can contain config keys the installed peft
# rejects ("unexpected keyword argument ..."). Drop those, and point the adapter at the
# hub base model instead of a Colab-local path that no longer exists.
cfg_path = os.path.join(LOCAL_ADAPTER, "adapter_config.json")
with open(cfg_path) as f:
    cfg = json.load(f)
allowed = set(inspect.signature(LoraConfig.__init__).parameters) | {"peft_type", "auto_mapping"}
dropped = sorted(k for k in cfg if k not in allowed)
for k in dropped:
    cfg.pop(k)
cfg["base_model_name_or_path"] = BASE_MODEL
cfg["inference_mode"] = True
with open(cfg_path, "w") as f:
    json.dump(cfg, f, indent=2)
if dropped:
    print("Removed adapter config keys unsupported by this peft version:", dropped)

# Use the tokenizer saved with the adapter if there is one (keeps any added tokens / chat template).
tok_src = LOCAL_ADAPTER if os.path.isfile(os.path.join(LOCAL_ADAPTER, "tokenizer_config.json")) else BASE_MODEL
tokenizer = AutoTokenizer.from_pretrained(tok_src)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("tokenizer from:", tok_src)

dtype_kw = {"dtype": dtype} if version.parse(transformers.__version__) >= version.parse("4.56") else {"torch_dtype": dtype}
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **dtype_kw).to(device)

# Only grow embeddings if the tokenizer has more tokens than the model (Qwen pads its
# embedding matrix, so shrinking it would break adapters trained on the default size).
if len(tokenizer) > model.get_input_embeddings().weight.shape[0]:
    model.resize_token_embeddings(len(tokenizer))
    print("Resized embeddings to", len(tokenizer))

try:
    model = PeftModel.from_pretrained(model, LOCAL_ADAPTER)
except Exception as e:
    raise RuntimeError(
        f"Adapter failed to load ({type(e).__name__}: {e}). A 'size mismatch' means the adapter was trained "
        "on a different base model or vocabulary size than BASE_MODEL."
    ) from None

model = model.merge_and_unload()  # faster inference, no peft overhead per request
model.eval()

GEN_LOCK = threading.Lock()  # one generation at a time; the model is not thread-safe

def generate_reply(message: str) -> str:
    messages = ([{"role": "system", "content": SYSTEM_PROMPT}] if SYSTEM_PROMPT else []) + [{"role": "user", "content": message}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with GEN_LOCK, torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("Smoke test:", generate_reply("What is a Python list? Answer in one sentence.")[:300])

In [ ]:
# 5. Start the FastAPI server (stops any previous instance first)
import hmac, os, subprocess, time
import requests, uvicorn
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel

app = FastAPI(title="Study Buddy Model API")

class ChatRequest(BaseModel):
    message: str

@app.get("/health")
def health():
    return {"status": "ok", "model": BASE_MODEL, "device": device}

@app.post("/generate")
def generate(req: ChatRequest, authorization: str = Header(default="")):
    # Constant-time comparison; the token is read from Colab Secrets in cell 2.
    if not hmac.compare_digest(authorization.encode(), f"Bearer {API_TOKEN}".encode()):
        raise HTTPException(status_code=401, detail="Unauthorized")
    message = req.message.strip()
    if not message or len(message) > MAX_MESSAGE_CHARS:
        raise HTTPException(status_code=400, detail=f"message must be 1-{MAX_MESSAGE_CHARS} characters")
    try:
        reply = generate_reply(message)
    except Exception as e:
        print("Generation error:", type(e).__name__)
        raise HTTPException(status_code=500, detail="Generation failed")
    if not reply:
        raise HTTPException(status_code=500, detail="Model returned an empty reply")
    return {"reply": reply}

# An old server keeps the token it started with, so a stale one causes 401s after a
# secret change. Stop ours if this cell ran before...
import socket

def port_in_use(port):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

if "SERVER" in globals():
    SERVER.should_exit = True
    for _ in range(20):
        if not port_in_use(PORT):
            break
        time.sleep(0.5)

# ...and any other process still holding the port.
holders = subprocess.run(["fuser", f"{PORT}/tcp"], capture_output=True, text=True).stdout.split()
for pid in holders:
    if int(pid) == os.getpid():
        raise RuntimeError(f"Port {PORT} is held by this notebook's own kernel from an older cell. Use Runtime → Restart session, then Run all.")
    subprocess.run(["kill", "-9", pid])
    print("Stopped old process on port", PORT, "pid", pid)
time.sleep(1)
if port_in_use(PORT):
    raise RuntimeError(f"Port {PORT} is still in use. Use Runtime → Restart session, then Run all.")

# loop="asyncio": plain asyncio is the safest choice for a server running in a background thread
SERVER = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning", loop="asyncio"))
threading.Thread(target=SERVER.run, daemon=True).start()

# Wait for *this* server (not a stale one) to be up and answering.
for _ in range(30):
    if SERVER.started:
        try:
            if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).ok:
                print(f"Server running on port {PORT}")
                break
        except requests.RequestException:
            pass
    time.sleep(1)
else:
    raise RuntimeError("Server did not start within 30s — check the cell output above for errors")

In [ ]:
# 6. Open the ngrok tunnel
from pyngrok import ngrok, conf
from pyngrok.exception import PyngrokError

conf.get_default().auth_token = NGROK_AUTHTOKEN
ngrok.kill()  # close tunnels from earlier runs so only one points at this server

NGROK_HINTS = {
    "ERR_NGROK_108": "Another ngrok session is still running on your account (e.g. an old Colab tab). Stop it at dashboard.ngrok.com/agents, then rerun this cell.",
    "ERR_NGROK_4018": "The NGROK_AUTHTOKEN secret is not a valid ngrok authtoken.",
    "ERR_NGROK_105": "The NGROK_AUTHTOKEN secret is not a valid ngrok authtoken.",
    "ERR_NGROK_320": "NGROK_DOMAIN is not a domain on your ngrok account (check dashboard.ngrok.com/domains).",
}
try:
    tunnel = ngrok.connect(PORT, "http", domain=NGROK_DOMAIN) if NGROK_DOMAIN else ngrok.connect(PORT, "http")
except PyngrokError as e:
    hint = next((h for code, h in NGROK_HINTS.items() if code in str(e)), "")
    raise RuntimeError(f"ngrok failed to start. {hint}\n{str(e)[:300]}") from None
PUBLIC_URL = tunnel.public_url.replace("http://", "https://")

print("Put this in Backend/.env as STUDY_BUDDY_MODEL_URL:")
print(f"\n    {PUBLIC_URL}/generate\n")

In [ ]:
# 7. Self-test through the public URL (exactly what the backend will do)
H = {"ngrok-skip-browser-warning": "1"}
health = requests.get(f"{PUBLIC_URL}/health", headers=H, timeout=30)
no_auth = requests.post(f"{PUBLIC_URL}/generate", json={"message": "hi"}, headers=H, timeout=30)
ok = requests.post(
    f"{PUBLIC_URL}/generate",
    json={"message": "Explain recursion in two sentences."},
    headers={**H, "Authorization": f"Bearer {API_TOKEN}"},
    timeout=180,
)
print("health:", health.status_code, health.json())
print("without token:", no_auth.status_code, "(expected 401)")
print("with token:", ok.status_code, "(expected 200)")
reply = ok.json().get("reply") if ok.ok else None
assert isinstance(reply, str) and reply.strip(), f"No valid reply: {ok.text[:200]}"
print("reply:", reply[:300])
print("\n✅ Ready. Keep this tab open; the API stops when the runtime disconnects.")

## After it's running

1. If you don't use `NGROK_DOMAIN`, copy the URL from cell 6 into `STUDY_BUDDY_MODEL_URL` in `Backend/.env` and restart the backend.
2. In `Backend`, run `npm run ai:check` — it verifies the URL, the token match and a real reply without printing secrets.
3. In the app: AI Tutor → **Fine-tuned model**.

**Troubleshooting**
- `401` from `npm run ai:check`: the Colab secret and `Backend/.env` token differ — compare the lengths printed by cell 2 and `ai:check`, fix the Colab secret, then rerun cells 2, 5, 6.
- `ERR_NGROK_3200` / 404: the runtime disconnected — Run all again (and update the URL if it changed).
- Adapter `size mismatch`: the adapter was trained on a different base; set `BASE_MODEL` to that base.
- `No module named 'triton.ops'`: comes from an old bitsandbytes. Cell 1 removes it and cell 4 blocks it; if you still see it, use Runtime → Restart session, then Run all.